# Cape Jones — Living Room Scene Render (Colab GPU)

Renders `3d-assets/living_room.blend` **headless with Cycles on a Colab GPU** (T4 typical on the
free tier). Run cells top to bottom, or **Runtime → Run all**.

**Before you run:**
1. **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 if offered) → Save.
2. The repo `mhvnsnt/cape-jones` is **private**, so the scene download cell will ask for a
   GitHub personal access token (fine-grained, **Contents: read-only** on `cape-jones`).

**Colorway canon is LOCKED** (`3d-assets/living-room/SCENE-CANON.md`). This notebook only sets
render parameters (samples, resolution %, frame range, output path) and a GPU device — it never
touches materials, colors, lights, or geometry.

**Honest status (2026-10-16):** the JSON is validated and the full command chain was proven on a
local CPU run (Blender opened the .blend, rendered frame 1 to PNG, PNG viewed — not black). The
GPU cells are verified by construction (Blender 5.2 Cycles API + T4/OptiX compatibility). The
first **live Colab GPU run still needs one manual Run-all** — there is no Colab execution API.
See `docs/pipeline/COLAB.md` for limits and the Colab-vs-Kaggle decision guide.

In [ ]:
# Cell 1 — Prove the GPU is really there (free tier: T4 most common, NOT guaranteed)
import shutil, subprocess
if shutil.which("nvidia-smi"):
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True)
    print("GPU:", out.stdout.strip() or "(nvidia-smi returned nothing)")
    if out.returncode != 0:
        print("WARNING: nvidia-smi failed — is the runtime set to GPU?")
else:
    print("WARNING: no nvidia-smi — Runtime → Change runtime type → GPU, then re-run.")

In [ ]:
# Cell 2 — Install Blender 5.2.2 LTS (PINNED — verified 2026-10-16 on blender.org/download/lts/)
# ~383 MB tarball, ~1-2 min on Colab's network. Re-run is a no-op if already installed.
import os, subprocess
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"
BLENDER_DIR = "blender-5.2.2-linux-x64"
EXPECTED_BYTES = 383295504
if os.path.isdir(BLENDER_DIR) and os.path.isfile(os.path.join(BLENDER_DIR, "blender")):
    print("Blender already installed, skipping download.")
else:
    subprocess.run(["wget", "--show-progress", "-O", "blender.tar.xz", BLENDER_URL], check=True)
    size = os.path.getsize("blender.tar.xz")
    assert size == EXPECTED_BYTES, f"truncated download: {size} != {EXPECTED_BYTES} bytes"
    print(f"tarball OK ({size} bytes), extracting…")
    subprocess.run(["tar", "-xf", "blender.tar.xz"], check=True)
subprocess.run([os.path.join(BLENDER_DIR, "blender"), "--version"], check=True)

In [ ]:
# Cell 3 — Download living_room.blend from origin/main
# NOTE: mhvnsnt/cape-jones is PRIVATE, so the raw URL 404s without auth.
# Tries anonymous first (works if the repo is ever made public), then asks for a PAT.
import getpass, os, urllib.request, urllib.error
SCENE_URL = "https://raw.githubusercontent.com/mhvnsnt/cape-jones/main/3d-assets/living_room.blend"
SCENE_FILE = "living_room.blend"

def fetch(url, token=None):
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=180) as r, open(SCENE_FILE, "wb") as f:
        f.write(r.read())

try:
    fetch(SCENE_URL)
    print("downloaded anonymously (repo is public to this network)")
except urllib.error.HTTPError as e:
    if e.code != 404:
        raise
    print("raw URL returned 404 → the repo is private; a token is required.")
    token = getpass.getpass(
        "GitHub PAT (fine-grained, Contents: read-only on mhvnsnt/cape-jones): ").strip()
    assert token, "no token pasted — cannot download the scene"
    fetch(SCENE_URL, token)
    print("downloaded with token auth")

with open(SCENE_FILE, "rb") as f:
    magic = f.read(12)
assert magic.startswith(b"BLENDER"), f"not a .blend file (magic={magic!r}) — check the URL/token"
print(f"OK: {SCENE_FILE} ({os.path.getsize(SCENE_FILE)} bytes, {magic.decode()})")

In [ ]:
# Cell 4 — Render parameters (edit these, then Runtime → Run all)
FRAME_START = 1
FRAME_END   = 1      # smoke test: single frame. Full shot = 1-360 (15 s @ 24 fps)
SAMPLES     = 64     # smoke-test quality; final look = 128-256
RES_PCT     = 50     # scene default: 960x540 @ 50% = 480x270. 100 = full 960x540
OUT_DIR     = "/content/cape-jones-renders"
print(f"frames {FRAME_START}-{FRAME_END}, {SAMPLES} samples, {RES_PCT}% -> {OUT_DIR}")

In [ ]:
%%writefile render_scene.py
"""Cape Jones living-room scene renderer (runs inside headless Blender).
Reads params from env: FRAME_START, FRAME_END, SAMPLES, RES_PCT, OUT_DIR.
CANON GUARD: sets render params + GPU device ONLY. Never touches materials,
colors, lights, or geometry (see 3d-assets/living-room/SCENE-CANON.md).
Usage: blender -b living_room.blend --python render_scene.py
"""
import bpy, os, time

def env_int(name, default):
    try:
        return int(os.environ.get(name, default))
    except (TypeError, ValueError):
        return default

FRAME_START = env_int("FRAME_START", 1)
FRAME_END   = env_int("FRAME_END", 1)
SAMPLES     = env_int("SAMPLES", 64)
RES_PCT     = env_int("RES_PCT", 50)
OUT_DIR     = os.environ.get("OUT_DIR", "/content/cape-jones-renders")
os.makedirs(OUT_DIR, exist_ok=True)

sc = bpy.context.scene

# --- GPU: prefer OPTIX (T4 is Turing, OptiX-supported) -> CUDA -> CPU fallback ---
prefs = bpy.context.preferences.addons["cycles"].preferences
for attr in ("refresh_devices", "get_devices"):  # API name differs across Blender versions
    if hasattr(prefs, attr):
        try:
            getattr(prefs, attr)()
            break
        except Exception as e:
            print(f"device refresh via {attr} failed: {e}")

gpu_done = False
for dev_type in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dev_type
    except TypeError:
        continue  # backend not offered by this Blender build
    for attr in ("refresh_devices", "get_devices"):
        if hasattr(prefs, attr):
            try:
                getattr(prefs, attr)()
            except Exception:
                pass
    gpus = [d for d in prefs.devices if d.type == dev_type]
    if gpus:
        for d in prefs.devices:
            d.use = (d in gpus)
        sc.cycles.device = "GPU"
        print(f"RENDER DEVICE: GPU via {dev_type}: {[d.name for d in gpus]}")
        gpu_done = True
        break
if not gpu_done:
    sc.cycles.device = "CPU"
    print("RENDER DEVICE: CPU fallback (no CUDA/OPTIX GPU found)")

# --- render params ---
sc.cycles.samples = SAMPLES
sc.render.resolution_percentage = RES_PCT
sc.frame_start = FRAME_START
sc.frame_end = FRAME_END
sc.render.image_settings.file_format = "PNG"
sc.render.filepath = os.path.join(OUT_DIR, "frame_####")
rx = sc.render.resolution_x * RES_PCT // 100
ry = sc.render.resolution_y * RES_PCT // 100
print(f"RENDERING frames {FRAME_START}-{FRAME_END}, {SAMPLES} samples, {rx}x{ry} -> {OUT_DIR}")

t0 = time.time()
bpy.ops.render.render(animation=True)
print(f"DONE in {time.time() - t0:.1f}s")

In [ ]:
# Cell 6 — Render headless (Blender's exit code is preserved; full log -> render.log)
import os, subprocess
env = {**os.environ,
       "FRAME_START": str(FRAME_START), "FRAME_END": str(FRAME_END),
       "SAMPLES": str(SAMPLES), "RES_PCT": str(RES_PCT), "OUT_DIR": OUT_DIR}
r = subprocess.run(["./blender-5.2.2-linux-x64/blender", "-b", "living_room.blend",
                    "--python", "render_scene.py"],
                   env=env, capture_output=True, text=True)
open("render.log", "w").write(r.stdout + r.stderr)
print("\n".join((r.stdout + r.stderr).splitlines()[-25:]))
r.check_returncode()  # raises with the log above if Blender failed

In [ ]:
# Cell 7 — Verify outputs, preview, download
import glob, os
from IPython.display import Image, display
pngs = sorted(glob.glob(os.path.join(OUT_DIR, "frame_*.png")))
assert pngs, "no PNGs rendered — inspect render.log"
empty = [p for p in pngs if os.path.getsize(p) == 0]
assert not empty, f"empty PNGs: {empty} — inspect render.log"
print(f"{len(pngs)} frame(s) OK — first: {pngs[0]} ({os.path.getsize(pngs[0])} bytes)")
display(Image(pngs[0], width=640))
if len(pngs) > 1:
    display(Image(pngs[-1], width=640))
    print("(last frame shown above)")
!zip -q -j renders.zip {OUT_DIR}/frame_*.png && ls -lh renders.zip
from google.colab import files
files.download("renders.zip")

## Done — what next

- **More frames:** edit Cell 4 (`FRAME_END = 360`, `SAMPLES = 128`) and **Runtime → Run all**
  again. Long runs: keep the tab active — the free tier disconnects idle runtimes (~90 min).
- **Canon check:** renders must show coral-pink furniture + warm oak + violet `#731AE5` LED wash
  (`3d-assets/living-room/SCENE-CANON.md`). This notebook never edits the scene; if a render
  looks off-canon, the `.blend` on `main` changed — check `git log` on `3d-assets/living_room.blend`.
- **Batch/overnight renders:** use the Kaggle GPU path instead — headless, 30 GPU-hr/week,
  agent-drivable via API. See `docs/research/FREE-3D-TOOLS.md` §1a and `docs/pipeline/COLAB.md`.
- **Status:** first live Colab GPU run pending — one manual Run-all (owner or browser lane).